### 1: Imports

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

torch.set_num_threads(1)
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('imports ok')

imports ok


### 2: Load Data

In [48]:
data_path = os.path.join(BASE_DIR, 'processed_data.csv')
df = pd.read_csv(data_path)

feature_cols = [col for col in df.columns if col not in ['pain_level', 'person_id']]
input_size = len(feature_cols)
people_ids = df['person_id'].unique()

print(f'Total windows: {len(df)}')
print(f'People: {people_ids}')
print(f'Features: {input_size}')
print(f'\nClass balance:')
print(df.groupby(['person_id', 'pain_level']).size().unstack(fill_value=0))

# split person 0's data 80/20 for demo-focused evaluation
person0_df = df[df['person_id'] == 0]
others_df  = df[df['person_id'] != 0]

p0_train, p0_test = train_test_split(
    person0_df, test_size=0.2, random_state=42, stratify=person0_df['pain_level']
)

# training pool: everyone else + 80% of person 0
train_pool = pd.concat([others_df, p0_train], ignore_index=True)
test_pool  = p0_test

X_train_p0 = train_pool[feature_cols].values
y_train_p0 = train_pool['pain_level'].values.astype(int)
X_test_p0  = test_pool[feature_cols].values
y_test_p0  = test_pool['pain_level'].values.astype(int)

print(f'\nPerson 0 train samples: {len(p0_train)}')
print(f'Person 0 test samples:  {len(p0_test)}')
print(f'Total training pool:    {len(train_pool)}')

X      = df[feature_cols].values
y      = df['pain_level'].values.astype(int)
groups = df['person_id'].values

Total windows: 1561
People: [0 1 2]
Features: 15

Class balance:
pain_level    0    1    2    3
person_id                     
0           206  139  260  219
1            84  144   84   75
2            83  114   73   80

Person 0 train samples: 659
Person 0 test samples:  165
Total training pool:    1396


### 3: helper funcs

In [49]:
def print_confusion_matrix(true, pred, title, num_classes=4):
    cm = confusion_matrix(true, pred)
    print(f'\n{title}')
    for i, row in enumerate(cm):
        print(f'true L{i}: {row.tolist()}')

class PainDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        labels = y.copy().astype(int)
        if labels.min() == 1:
            labels = labels - 1
        self.y = torch.tensor(labels, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        features = self.X[idx]
        if self.augment:
            features = features + torch.randn_like(features) * 0.01
        return features, self.y[idx]

class PainTrackerMLP(nn.Module):
    def __init__(self, input_size, num_classes=4):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 64)
        self.relu1  = nn.ReLU()
        self.drop1  = nn.Dropout(0.1)
        self.layer2 = nn.Linear(64, 32)
        self.relu2  = nn.ReLU()
        self.drop2  = nn.Dropout(0.1)
        self.output = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.drop1(self.relu1(self.layer1(x)))
        x = self.drop2(self.relu2(self.layer2(x)))
        return self.output(x)

EPOCHS_BASE   = 100
EPOCHS_FINETUNE = 50
BATCH_SIZE    = 16
LR_BASE       = 0.001
LR_FINETUNE   = 0.0003  # lower lr for fine-tuning
NUM_CLASSES   = 4

print('helpers and model defined')

helpers and model defined


### 4: SVM LOOCV

In [50]:
scaler_svm = StandardScaler()
X_tr_svm   = scaler_svm.fit_transform(X_train_p0)
X_te_svm   = scaler_svm.transform(X_test_p0)

weights_svm = compute_sample_weight('balanced', y_train_p0)
svm = SVC(kernel='rbf', C=5, gamma='scale')
svm.fit(X_tr_svm, y_train_p0, sample_weight=weights_svm)
svm_preds = svm.predict(X_te_svm)

svm_acc = accuracy_score(y_test_p0, svm_preds) * 100
print(f'SVM Accuracy on Person 0: {svm_acc:.2f}%')
print_confusion_matrix(y_test_p0, svm_preds, 'SVM Confusion Matrix')

SVM Accuracy on Person 0: 76.36%

SVM Confusion Matrix
true L0: [34, 3, 4, 0]
true L1: [4, 17, 3, 4]
true L2: [4, 1, 43, 4]
true L3: [2, 3, 7, 32]


### 5: XGBoost

In [51]:
scaler_xgb = StandardScaler()
X_tr_xgb   = scaler_xgb.fit_transform(X_train_p0)
X_te_xgb   = scaler_xgb.transform(X_test_p0)

weights_xgb = compute_sample_weight('balanced', y_train_p0)
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=1
)
xgb_model.fit(X_tr_xgb, y_train_p0, sample_weight=weights_xgb)
xgb_preds = xgb_model.predict(X_te_xgb)

xgb_acc = accuracy_score(y_test_p0, xgb_preds) * 100
print(f'XGBoost Accuracy on Person 0: {xgb_acc:.2f}%')
print_confusion_matrix(y_test_p0, xgb_preds, 'XGBoost Confusion Matrix')

XGBoost Accuracy on Person 0: 92.12%

XGBoost Confusion Matrix
true L0: [39, 0, 1, 1]
true L1: [0, 22, 6, 0]
true L2: [0, 0, 49, 3]
true L3: [1, 0, 1, 42]


In [52]:
# simulate the real product flow:
# 1. base model already trained on all people (from cell 5)
# 2. new user (person 0) logs pain naturally over time
# 3. once enough events accumulate, warm-start fine-tune on their data

# use person 0 train data as the simulated natural logs
X_p0_natural = scaler_xgb.transform(p0_train[feature_cols].values)
y_p0_natural = p0_train["pain_level"].values.astype(int)

# warm-start: add 20 new trees on top of the base model
# these new trees correct for person 0 specifically
# without overwriting the general patterns from the base
personalized_model = XGBClassifier(
    n_estimators=20,
    max_depth=3,
    learning_rate=0.05,  # lower lr - nudge not overwrite
    eval_metric="mlogloss",
    verbosity=0,
    n_jobs=1
)
personalized_model.fit(
    X_p0_natural, y_p0_natural,
    xgb_model=xgb_model.get_booster()  # start from base model trees
)

# evaluate personalized model on person 0 test set
personalized_preds = personalized_model.predict(X_te_xgb)
personalized_acc = accuracy_score(y_test_p0, personalized_preds) * 100

print(f"Base XGBoost accuracy on Person 0:         {xgb_acc:.2f}%")
print(f"Personalized XGBoost accuracy on Person 0:  {personalized_acc:.2f}%")
print(f"Improvement: {personalized_acc - xgb_acc:+.2f}%")
print_confusion_matrix(y_test_p0, personalized_preds, "Personalized XGBoost Confusion Matrix")

# save personalized model separately
personalized_model.save_model(os.path.join(BASE_DIR, "personalized_model_p0.json"))
print("personalized_model_p0.json saved.")
print("In the real app, this file would live on the user phone and update as they log more data.")

Base XGBoost accuracy on Person 0:         92.12%
Personalized XGBoost accuracy on Person 0:  95.76%
Improvement: +3.64%

Personalized XGBoost Confusion Matrix
true L0: [40, 0, 1, 0]
true L1: [0, 24, 4, 0]
true L2: [0, 0, 51, 1]
true L3: [0, 0, 1, 43]
personalized_model_p0.json saved.
In the real app, this file would live on the user phone and update as they log more data.


### 7: MLP

In [ ]:
# --- step 1: train base model on everyone else's data ---
scaler_mlp = StandardScaler()

X_others = scaler_mlp.fit_transform(others_df[feature_cols].values)
y_others = others_df['pain_level'].values.astype(int)

base_loader = DataLoader(
    PainDataset(X_others, y_others, augment=True),
    batch_size=BATCH_SIZE, shuffle=True
)

base_model = PainTrackerMLP(input_size=input_size, num_classes=NUM_CLASSES)
optimizer_base = optim.Adam(base_model.parameters(), lr=LR_BASE, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

base_model.train()
for epoch in range(EPOCHS_BASE):
    for bX, by in base_loader:
        optimizer_base.zero_grad()
        loss = criterion(base_model(bX), by)
        loss.backward()
        optimizer_base.step()

print('base model trained on other people')

# --- step 2: freeze layer1 and layer2, only fine-tune output layer on person 0 ---
for param in base_model.layer1.parameters():
    param.requires_grad = False
for param in base_model.layer2.parameters():
    param.requires_grad = False
# output layer stays trainable

X_p0_train_scaled = scaler_mlp.transform(p0_train[feature_cols].values)
y_p0_train = p0_train['pain_level'].values.astype(int)

finetune_loader = DataLoader(
    PainDataset(X_p0_train_scaled, y_p0_train, augment=True),
    batch_size=BATCH_SIZE, shuffle=True
)

# only pass trainable params to optimizer
optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, base_model.parameters()),
    lr=LR_FINETUNE, weight_decay=1e-4
)

base_model.train()
for epoch in range(EPOCHS_FINETUNE):
    for bX, by in finetune_loader:
        optimizer_ft.zero_grad()
        loss = criterion(base_model(bX), by)
        loss.backward()
        optimizer_ft.step()

print('fine-tuned on person 0 training data')

# --- step 3: evaluate on person 0 test set ---
X_p0_test_scaled = scaler_mlp.transform(p0_test[feature_cols].values)
y_p0_test = p0_test['pain_level'].values.astype(int)

test_loader_mlp = DataLoader(
    PainDataset(X_p0_test_scaled, y_p0_test, augment=False),
    batch_size=BATCH_SIZE, shuffle=False
)

base_model.eval()
mlp_true, mlp_pred = [], []
with torch.no_grad():
    for bX, by in test_loader_mlp:
        _, predicted = torch.max(base_model(bX), 1)
        mlp_true.extend(by.tolist())
        mlp_pred.extend(predicted.tolist())

mlp_acc = accuracy_score(mlp_true, mlp_pred) * 100
print(f'\nMLP Transfer Learning Accuracy on Person 0: {mlp_acc:.2f}%')
print_confusion_matrix(mlp_true, mlp_pred, 'MLP Transfer Learning Confusion Matrix')

### 8: compare

In [ ]:
print('========= Model Accuracy on Person 0 (Demo Target) =========')
print(f'SVM:                    {svm_acc:.2f}%')
print(f'XGBoost:                {xgb_acc:.2f}%')
print(f'MLP Transfer Learning:  {mlp_acc:.2f}%')
print('============================================================')

========= Model Accuracy on Person 0 (Demo Target) =========
SVM:                    76.36%
XGBoost:                92.12%


### 9: save best model

In [ ]:
# pick whichever model performed best on person 0
best_acc = max(svm_acc, xgb_acc, mlp_acc)
best_name = ['SVM', 'XGBoost', 'MLP'][np.argmax([svm_acc, xgb_acc, mlp_acc])]
print(f'Best model: {best_name} at {best_acc:.2f}%')

# save XGBoost with 3x person 0 weighting as the production model
# (XGBoost is easiest to deploy on Android via ONNX or direct inference)
sample_weights_final = np.where(df['person_id'].values == 0, 3.0, 1.0)

final_scaler = StandardScaler()
X_all = final_scaler.fit_transform(X)

final_model = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=1
)
final_model.fit(X_all, y, sample_weight=sample_weights_final)

final_model.save_model(os.path.join(BASE_DIR, 'base_model.json'))
joblib.dump(final_scaler, os.path.join(BASE_DIR, 'scaler.pkl'))

# also save the fine-tuned MLP in case it's needed
torch.save(base_model.state_dict(), os.path.join(BASE_DIR, 'mlp_finetuned.pth'))
joblib.dump(scaler_mlp, os.path.join(BASE_DIR, 'scaler_mlp.pkl'))

print('saved: base_model.json, scaler.pkl (XGBoost)')
print('saved: mlp_finetuned.pth, scaler_mlp.pkl (MLP)')

Best model: XGBoost at 92.12%
saved: base_model.json, scaler.pkl (XGBoost)
saved: mlp_finetuned.pth, scaler_mlp.pkl (MLP)
